# Module 04: Web Search & Browsing

Agents trained on static data cannot answer questions like "What happened this week?" or "What is the current price of X?" — their knowledge has a hard cutoff date.

Web tools solve this by giving agents the ability to retrieve live information on demand:

- **`DuckDuckGoSearchTool`** — runs a web search and returns a list of results (title, URL, snippet)
- **`VisitWebpageTool`** — fetches a URL and returns the page content as cleaned markdown text

Together they enable the **research pattern**:

```
search → select best URL → visit → extract → reason
```

**Warning:** web content is noisy. Agents need clear, specific task prompts to extract the signal they need from the noise they will encounter.

## Setup

In [ ]:
# Install required packages
# Uncomment the line below if running in Google Colab or a fresh environment
# !uv pip install smolagents python-dotenv duckduckgo-search mlflow
# Or using pip:
# !pip install smolagents python-dotenv duckduckgo-search mlflow

In [ ]:
import os

# ----- HF_TOKEN Setup -----
# Option A: Load from .env file (local development)
# from dotenv import load_dotenv
# load_dotenv()

# Option B: Google Colab Secrets
# Uncomment the lines below when running in Google Colab.
# Go to: Colab → Secrets (🔑 icon) → Add HF_TOKEN
# from google.colab import userdata
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Option C: Set directly (not recommended for shared notebooks)
# os.environ['HF_TOKEN'] = 'hf_your_token_here'

In [ ]:
import os
import time
from dotenv import load_dotenv
from smolagents import CodeAgent, InferenceClientModel, DuckDuckGoSearchTool, VisitWebpageTool

load_dotenv()

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    token=os.environ["HF_TOKEN"],
)
print("Ready.")

## DuckDuckGoSearchTool

Let's first use the tool directly (without an agent) to understand what it returns.

In [ ]:
search = DuckDuckGoSearchTool()

# Direct call — returns a string of formatted results
results = search("smolagents huggingface 2024")
print(results[:2000])

## VisitWebpageTool

Visits a URL and returns the page content as cleaned markdown text.

In [ ]:
visit = VisitWebpageTool()

content = visit("https://huggingface.co/blog/smolagents")
print(content[:2000])

## Search-Only Research Agent

An agent that can search but not visit pages — good for quick queries.

In [ ]:
research_agent = CodeAgent(
    tools=[DuckDuckGoSearchTool()],
    model=model,
    max_steps=5,
)

result = research_agent.run(
    "What are the main differences between smolagents and LangChain? "
    "Search for recent comparisons and summarize the top 3 differences."
)
print("\nResult:", result)

## Deep Research Agent: Search + Visit

Now we give the agent both tools. It can search for relevant URLs, then visit the most promising ones to extract detailed information.

In [ ]:
deep_agent = CodeAgent(
    tools=[DuckDuckGoSearchTool(), VisitWebpageTool()],
    model=model,
    max_steps=8,
)

result = deep_agent.run(
    "Search for the latest HuggingFace blog post about AI agents. "
    "Visit the page and give me: (1) the title, (2) publication date if visible, "
    "(3) three key takeaways in bullet points."
)
print("\nResult:", result)

## Reliability Best Practices

Web-enabled agents introduce new failure modes. Here's how to guard against them.

### 1. Ask for citations
Always include 'provide the source URL' in your task prompt.

### 2. Limit max_steps
Web research can spiral. 5–8 steps is usually enough; more rarely helps.

### 3. Be specific in your task
Vague tasks → unfocused searches → low-quality results.

### 4. Verify critical information
For factual claims, check the agent's cited sources yourself.

In [ ]:
# Note how we explicitly ask for the source URL
verifiable_agent = CodeAgent(
    tools=[DuckDuckGoSearchTool(), VisitWebpageTool()],
    model=model,
    max_steps=6,
)

result = verifiable_agent.run(
    "Find the current stable version of Python. "
    "Visit python.org to confirm. "
    "Return the version number AND the URL where you found it."
)
print("\nResult:", result)

## Exercises

In [ ]:
# TODO Exercise 1: Python release finder
# Build an agent with DuckDuckGoSearchTool + VisitWebpageTool
# Task: "What is the latest stable release of Python? 
#        Visit python.org/downloads and extract the exact version number."
# After running:
#   - Print the result
#   - Print how many steps were used
#   - Note: did the agent visit a page or just search?

# Your code here:

In [ ]:
# TODO Exercise 2: Data engineering research agent
# Build an agent with both web tools and max_steps=10
# Task: "Search for 3 recent articles (2023 or later) about dbt (data build tool) 
#        best practices. For each article, visit the page and extract 
#        the top recommendation. Return a comparison table with columns:
#        Article Title | Top Recommendation | Source URL"
#
# Challenge: Does the agent reliably visit 3 separate URLs?
# If not, what would you change in the prompt to help it?

# Your code here:

## What You Built

You now know how to:
- Use `DuckDuckGoSearchTool` standalone and inside an agent
- Use `VisitWebpageTool` to read live web pages
- Build a search → visit → extract → reason research workflow
- Guard against hallucination with citation-requesting prompts
- Set appropriate `max_steps` for web research tasks

**Key insight:** Web tools make agents dramatically more useful, but also less predictable. The quality of your task description is even more important — vague tasks produce unfocused searches and poor results.

## Next Module Preview

**Module 05: Multi-Agent Orchestration**

A single agent with many tools can become unwieldy. In Module 05 you'll build a *system* of agents: a manager that delegates to specialist sub-agents, each focused on one job. This is the architecture behind production-grade agent systems.